# 5) Using the model — Exercises

**Goals:** split costs correctly and compute abatement costs, measure an interaction between two measures, predict by hand the CO₂ price at which emissions jump and confirm it with the model, and turn the model's running-cost savings into break-even investment costs.

The setup cell holds the lecture's year and its `build` / `evaluate` functions. Each `evaluate` call solves a full year (a few seconds). Money in MEUR per year unless stated otherwise, emissions in Mt CO₂.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import numpy as np
import pandas as pd
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- the lecture's synthetic year and model (same seed, same numbers) ----
rng = np.random.default_rng(7)
idx = pd.date_range("2025-01-01", periods=8760, freq="h")
hour, doy = idx.hour.to_numpy(), idx.dayofyear.to_numpy()
load = (2050 + 350 * np.cos(2 * np.pi * (doy - 15) / 365) + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0)
        - 150 * (idx.dayofweek.to_numpy() >= 5) + rng.normal(0, 60, 8760))
w = np.zeros(8760)
shocks = rng.normal(0, 1, 8760)
for t in range(1, 8760):
    w[t] = 0.97 * w[t - 1] + 0.24 * shocks[t]
wind_cf = np.clip(0.27 + 0.18 * w + 0.06 * np.cos(2 * np.pi * (doy - 15) / 365), 0, 0.95)
sun = np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * (0.65 + 0.35 * np.sin(2 * np.pi * (doy - 80) / 365))
cloud = np.clip(1 - 0.5 * rng.beta(2, 3, 365).repeat(24), 0.2, 1)
solar_cf = np.clip(sun * cloud * 0.95, 0, 1)
ror_cf = np.clip(0.55 + 0.25 * np.sin(2 * np.pi * (doy - 60) / 365), 0.2, 0.9)

EMISSIONS = {"coal": 0.34 / 0.38, "ccgt": 0.20 / 0.55, "imports": 0.20 / 0.55, "ocgt": 0.20 / 0.38}
BASE = {"solar": 1000, "coal": 600, "battery_mwh": 1200, "co2": 70.0, "gas": 35.0}


def build(solar=1000, coal=600, battery_mwh=1200, co2=70.0, gas=35.0):
    mc = lambda fuel, eff, ef: (fuel + co2 * ef) / eff
    n = pypsa.Network()
    n.set_snapshots(idx)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=load)

    def gen(name, p_nom, p_max_pu, cost, ramp=None):
        if p_nom > 0:
            limits = {} if ramp is None else {"ramp_limit_up": ramp / p_nom, "ramp_limit_down": ramp / p_nom}
            n.add("Generator", name, bus="HR", p_nom=p_nom, p_max_pu=p_max_pu, marginal_cost=cost, **limits)

    gen("solar", solar, solar_cf, 0.0)
    gen("wind", 1000, wind_cf, 0.0)
    gen("run_of_river", 500, ror_cf, 0.0)
    gen("nuclear_share", 350, 1.0, 12.0)
    gen("coal", coal, 1.0, mc(12, 0.38, 0.34), ramp=60)
    gen("ccgt", 900, 1.0, mc(gas, 0.55, 0.20), ramp=300)
    gen("imports", 800, 1.0, mc(gas, 0.55, 0.20) + 15)
    gen("ocgt", 400, 1.0, mc(gas, 0.38, 0.20))
    if battery_mwh > 0:
        n.add("StorageUnit", "battery", bus="HR", p_nom=300, max_hours=battery_mwh / 300,
              efficiency_store=0.95, efficiency_dispatch=0.95, cyclic_state_of_charge=True)
    return n


def evaluate(**assumptions):
    """Solve one year; money in MEUR/year (as in the lecture)."""
    a = {**BASE, **assumptions}
    n = build(**a)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    price, gen = n.buses_t.marginal_price["HR"], n.generators_t.p
    emissions = sum((gen[g] * f).sum() for g, f in EMISSIONS.items() if g in gen)
    return {"system_cost": n.objective / 1e6, "carbon_payments": a["co2"] * emissions / 1e6,
            "resource_cost": n.objective / 1e6 - a["co2"] * emissions / 1e6,
            "consumer_cost": (price * load).sum() / 1e6, "emissions_Mt": emissions / 1e6}

### Warm-ups

**1. Resource cost and abatement cost**

Write `resource_cost(system_cost, co2_price, emissions_Mt)` (MEUR) and `abatement_cost(base, scenario)` (€/t) — the change in resource cost per tonne avoided, from two result dicts. Return `None` if emissions do not fall. Check them on the lecture's coal-exit numbers.

In [2]:
def resource_cost(system_cost: float, co2_price: float, emissions_Mt: float) -> float:
    """System cost without the carbon payments (MEUR)."""
    ### BEGIN SOLUTION
    return system_cost - co2_price * emissions_Mt
    ### END SOLUTION


def abatement_cost(base: dict, scenario: dict):
    """Change in resource cost per tonne of CO2 avoided (EUR/t), or None if emissions do not fall."""
    ### BEGIN SOLUTION
    avoided = base["emissions_Mt"] - scenario["emissions_Mt"]
    if avoided <= 1e-12:
        return None
    return (scenario["resource_cost"] - base["resource_cost"]) / avoided
    ### END SOLUTION

In [3]:
# the lecture's numbers (MEUR/year, Mt): base and coal exit
lecture_base = {"resource_cost": 547.683, "emissions_Mt": 4.259}
lecture_coal_exit = {"resource_cost": 635.941, "emissions_Mt": 3.257}
assert np.isclose(resource_cost(845.789, 70.0, 4.259), 547.659, atol=0.05)
assert np.isclose(abatement_cost(lecture_base, lecture_coal_exit), 88.1, atol=0.1), "88 EUR/t, as in the lecture"
assert abatement_cost(lecture_base, {"resource_cost": 500.0, "emissions_Mt": 4.5}) is None, "emissions rose"

**2. Why "18 MEUR" is the wrong headline**

The coal exit raised the *system* cost by only 18.1 MEUR. Show with `resource_cost` that this equals the resource-cost increase minus the carbon payments no longer made, and write one sentence on why the resource cost is the right basis for comparing policies.

In [4]:
system_change = 863.931 - 845.789                                  # lecture: coal exit vs base, system cost
resource_change = lecture_coal_exit["resource_cost"] - lecture_base["resource_cost"]
### BEGIN SOLUTION
avoided_payments = 70.0 * (lecture_base["emissions_Mt"] - lecture_coal_exit["emissions_Mt"])
### END SOLUTION
assert np.isclose(system_change, resource_change - avoided_payments, atol=0.1)
print(f"system cost +{system_change:.1f} = resource cost +{resource_change:.1f} - carbon payments avoided {avoided_payments:.1f} MEUR")

system cost +18.1 = resource cost +88.3 - carbon payments avoided 70.1 MEUR


<!-- BEGIN SOLUTION -->
Carbon payments are a transfer from emitters to the state budget, not a use of resources: when the coal plant stops emitting, society does not save 70 MEUR — the treasury simply collects 70 MEUR less. Comparing policies on resource cost counts only what the economy actually gives up (fuel, operation), so a policy is not made to look cheap just because it reduces payments that were redistributed anyway.
<!-- END SOLUTION -->

### Core

**3. An interaction effect**

Write `battery_interaction()` that evaluates the four combinations of solar ∈ {1,000, 2,000} MW and battery ∈ {1,200, 2,400} MWh and returns a DataFrame of resource costs plus the **interaction term**: (saving from the larger battery with more solar) − (saving from the larger battery with today's solar). A positive interaction means the measures reinforce each other.

In [5]:
def battery_interaction() -> tuple[pd.DataFrame, float]:
    """Resource cost (MEUR/year) on the 2x2 grid, and the interaction term (MEUR/year)."""
    ### BEGIN SOLUTION
    grid = pd.DataFrame({b: {s: evaluate(solar=s, battery_mwh=b)["resource_cost"] for s in (1000, 2000)}
                         for b in (1200, 2400)})
    grid.index.name, grid.columns.name = "solar_MW", "battery_MWh"
    saving_today = grid.loc[1000, 1200] - grid.loc[1000, 2400]
    saving_more_solar = grid.loc[2000, 1200] - grid.loc[2000, 2400]
    return grid, saving_more_solar - saving_today
    ### END SOLUTION

In [6]:
grid, interaction = battery_interaction()
assert interaction > 1.0, "with more solar, the larger battery saves at least 1 MEUR/year more"
assert grid.loc[2000].max() < grid.loc[1000].min(), "more solar always lowers the running cost here"
print(f"interaction term: {interaction:.2f} MEUR/year")
grid.round(2)

interaction term: 1.89 MEUR/year


battery_MWh    1200    2400
solar_MW                   
1000         547.68  547.49
2000         471.57  469.48

**4. Predict an emissions jump by hand**

At today's CO₂ price (70 €/t) coal is cheaper than the imports, which are priced at the CCGT's cost plus 15 €/MWh. Write `switch_price()` returning the CO₂ price (€/t) at which coal's marginal cost equals the imports' — use M2.2's formula with coal (12 €/MWh_th, η 0.38, 0.34 t/MWh_th) and gas (35 €/MWh_th, η 0.55, 0.20 t/MWh_th). Then confirm with the model that emissions drop sharply between just below and just above that price.

In [7]:
def switch_price() -> float:
    """CO2 price (EUR/t) at which coal and the imports have equal marginal cost."""
    ### BEGIN SOLUTION
    coal_fuel, coal_carbon = 12 / 0.38, 0.34 / 0.38
    imports_fuel, imports_carbon = 35 / 0.55 + 15, 0.20 / 0.55
    return (imports_fuel - coal_fuel) / (coal_carbon - imports_carbon)
    ### END SOLUTION

In [8]:
c = switch_price()
assert np.isclose(c, 88.60, atol=0.01)
below, above = evaluate(co2=c - 3), evaluate(co2=c + 3)
assert below["emissions_Mt"] - above["emissions_Mt"] > 0.5, "emissions jump at the predicted CO2 price"
print(f"predicted switch at {c:.2f} EUR/t: emissions {below['emissions_Mt']:.2f} Mt at {c - 3:.1f} EUR/t, "
      f"{above['emissions_Mt']:.2f} Mt at {c + 3:.1f} EUR/t")

predicted switch at 88.60 EUR/t: emissions 4.17 Mt at 85.6 EUR/t, 3.27 Mt at 91.6 EUR/t


**5. Interpretation**

You predicted the jump with a one-line formula and the year-long model confirmed it. What does that tell you about where a carbon price "works", and why should a sensitivity analysis on the CO₂ price use more than two values?

<!-- BEGIN SOLUTION -->
A carbon price changes dispatch only when it reorders the merit order: between switching points, raising it only raises costs and prices, while at a switching point (here about 88.6 €/t, where coal falls behind the imports) emissions fall in one step. So a carbon price "works" in bands, and the size of the effect depends on how close the current price is to the next switch — for an existing system this is something you can predict from marginal costs before running any model, and then confirm. For sensitivity analysis it means that two values (low, high) can be badly misleading: they may both lie on the same side of a switch (showing no effect) or straddle one (suggesting a smooth response that does not exist). A sweep with enough points — or, better, a list of the switching prices computed by hand — shows the step structure that drives the policy result.
<!-- END SOLUTION -->

### Challenge

**6. What may the investments cost?**

The model counts no investment, so its savings are what the investments must pay for. Using your `grid` from task 3, write `break_even(saving_meur, capacity, rate, years)` returning the **annual** saving per unit of capacity and the **overnight** investment cost per unit at which the investment just breaks even (annuity factor r / (1 − (1 + r)⁻ⁿ)). Apply it to (a) the extra 1,000 MW of solar (per kW, 5 %, 25 years), and (b) the extra 1,200 MWh of battery, alongside the extra solar (per kWh, 5 %, 15 years).

In [9]:
def break_even(saving_meur: float, capacity: float, rate: float, years: int) -> tuple[float, float]:
    """Annual saving per unit of capacity and break-even overnight cost per unit (both in EUR)."""
    ### BEGIN SOLUTION
    annual = saving_meur * 1e6 / capacity
    annuity = rate / (1 - (1 + rate) ** -years)
    return annual, annual / annuity
    ### END SOLUTION

In [10]:
solar_saving = grid.loc[1000, 1200] - grid.loc[2000, 1200]                   # MEUR/year, extra 1 GW of solar
battery_saving = grid.loc[2000, 1200] - grid.loc[2000, 2400]                 # MEUR/year, extra 1,200 MWh with more solar
solar_annual, solar_overnight = break_even(solar_saving, 1_000_000, 0.05, 25)         # per kW
battery_annual, battery_overnight = break_even(battery_saving, 1_200_000, 0.05, 15)   # per kWh
assert np.isclose(break_even(1.0, 1.0, 0.05, 25)[1], 1e6 / 0.0709525, rtol=1e-4)
assert solar_overnight > 50 * battery_overnight, "the solar case is far stronger than the battery case"
print(f"solar:   {solar_annual:.1f} EUR/kW per year -> breaks even at {solar_overnight:,.0f} EUR/kW")
print(f"battery: {battery_annual:.2f} EUR/kWh per year -> breaks even at {battery_overnight:,.1f} EUR/kWh")

solar:   76.1 EUR/kW per year -> breaks even at 1,073 EUR/kW
battery: 1.74 EUR/kWh per year -> breaks even at 18.1 EUR/kWh


**7. Interpretation**

Is the extra solar likely to pay for itself from these savings? The extra battery? What would change the battery's answer — and what does this say about the brief's "running cost falls" headline?

<!-- BEGIN SOLUTION -->
The extra solar saves about 76 € per kW and year in running costs, which justifies an overnight investment of roughly 1,070 €/kW at 5 % over 25 years — in the range that utility-scale solar can plausibly reach, so its case looks sound in this system (a full study would check the cost assumptions, cannibalisation as solar grows, and network costs). The extra battery saves under 2 € per kWh and year, justifying only about 18 €/kWh — far below what battery storage costs today, so on these numbers it does not pay for itself. That would change in a system with more curtailment or wider price spreads: much more solar, less flexible plants (lesson 3's day), revenue from reserves and balancing, or scarcity prices — none of which this year contains. The lesson for the brief: a "running cost falls" headline is only half the answer; it must be set against what the measures cost to build, which is exactly what capacity-expansion models optimise (Module 05).
<!-- END SOLUTION -->